In [1]:
from google import genai
from pydantic import BaseModel, Field
from typing import List, Optional
import json

In [ ]:
json_path = "C:/Users/xnevi/Downloads/EateryAI/restaurants.menu_item_variations.json"
with open(json_path, encoding="utf-8") as f:
    json_object = json.load(f)
f.close()

##fstring input
json.dumps(json_object, indent=2)

In [2]:
### get a dummy "input.json" with a couple restaurants of data

json_path = "C:/Users/xnevi/Downloads/EateryAI/restaurants.menu_item_variations.json"
with open(json_path, encoding="utf-8") as f:
    json_object = json.load(f)
f.close()

subset_json = []
sample_restaurants = ["Chipotle"]
for item in json_object:
    if(item['restaurant_name'] in sample_restaurants):
        subset_json.append(item)

with open("subset_input.json", "w") as f:
    json.dump(subset_json, f)
f.close()

In [3]:
import json
import os
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

# 1. Define the desired JSON output structure using Pydantic
class CategorizedItem(BaseModel):
    item_id: str
    category_new: str = Field(
        description="Must be exactly one of: Entree, Side, Drink, Topping, Add-On"
    )

class CategorizationResult(BaseModel):
    items: list[CategorizedItem]

def categorize_menu_items(input_filepath: str, output_filepath: str):
    # Initialize the Gemini client (automatically picks up GEMINI_API_KEY from environment)
    client = genai.Client()
    
    # Read your input JSON data
    with open(input_filepath, encoding="utf-8") as f:
        menu_data = json.load(f)

    # 2. Define the instructions for the model
    system_instruction = """
    You are an expert at categorizing restaurant menu items. 
    Analyze the provided JSON list of menu items and assign exactly one of the following labels to the 'category_new' field:
    
    - Entree: Main items like burgers, burritos, bowls, pizzas, etc.
    - Side: Accompanying items like Fries, Chips, Side Salads, etc.
    - Drink: Beverages, sodas, shakes, water, etc.
    - Topping: Things added on to entrees for FREE (Infer from your internal knowledge, NOT the input data)
    - Add-On: Things added on to entrees that cost EXTRA (Infer from your internal knowledge, NOT the input data)
    
    Return a JSON object containing a list called 'items'. Each item should ONLY contain the 'item_id' and the assigned 'category_new'.
    """

    print("Sending data to Gemini for categorization...")
    
    # 3. Call the Gemini API, enforcing the JSON schema
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=json.dumps(menu_data),
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            response_mime_type="application/json",
            response_schema=CategorizationResult,
            temperature=0.1, # Low temperature for more consistent, factual categorization
        ),
    )

    # 4. Parse the response and save it to the output file
    try:
        # The response.text is guaranteed to be a JSON string matching our schema
        output_json = json.loads(response.text)
        
        with open(output_filepath, 'w') as f:
            json.dump(output_json["items"], f, indent=4)
            
        print(f"Success! Categorized {len(output_json['items'])} items. Saved to {output_filepath}")
        
    except Exception as e:
        print(f"An error occurred while parsing or saving: {e}")
        print("Raw response from model:")
        print(response.text)

# Example usage:
if __name__ == "__main__":
    # Create a dummy input file for testing
    '''
    dummy_data = [
        {"item_id": "101", "menu_item_name": "Double Cheeseburger", "restaurant_name": "Burger Joint", "price": 8.99},
        {"item_id": "102", "menu_item_name": "Large Fries", "restaurant_name": "Burger Joint", "price": 3.50},
        {"item_id": "103", "menu_item_name": "Ketchup Packets", "restaurant_name": "Burger Joint", "price": 0.00},
        {"item_id": "104", "menu_item_name": "Extra Bacon", "restaurant_name": "Burger Joint", "price": 1.50},
        {"item_id": "105", "menu_item_name": "Diet Cola", "restaurant_name": "Burger Joint", "price": 2.00}
    ]
    with open("input.json", "w") as f:
        json.dump(dummy_data, f)
    '''
    
    # Run the categorization
    #categorize_menu_items("input.json", "output.json")
    categorize_menu_items("subset_input.json", "output.json")

Sending data to Gemini for categorization...
Success! Categorized 99 items. Saved to output.json
